<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/QSAR/01_RecA_ChEMBL_EC50_Curation_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — RecA–TB ChEMBL EC50 Data Collection and Curation

> Clean senior-review version. This notebook contains only the final, logically connected pipeline.
> Exploratory/probing/failed scripts are intentionally excluded from the main workflow.

## Objective
Download/prepare RecA-related ChEMBL bioactivity data, standardize EC50 records, assign binary labels, and export the curated dataset for fingerprint generation.

In [1]:
# Install only when running in a fresh Colab environment
# !pip install -q pandas numpy scikit-learn chembl_webresource_client

## 1. Imports and project paths

In [2]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "raw"
CURATED_DIR = PROJECT_DIR / "outputs" / "recA_chembl" / "curated"

RAW_DIR.mkdir(parents=True, exist_ok=True)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAME = "RecA"
STANDARD_TYPE = "EC50"
ACTIVE_THRESHOLD_NM = 10000  # EC50 <= 10 µM = active-like

## 2. Load or download ChEMBL records

In [4]:
# If you already have a ChEMBL export, place it here:
!pip install -q chembl_webresource_client
RAW_FILE = RAW_DIR / "recA_chembl_raw_ec50.csv"

if RAW_FILE.exists():
    raw_df = pd.read_csv(RAW_FILE)
else:
    from chembl_webresource_client.new_client import new_client

    target = new_client.target
    activity = new_client.activity

    targets = target.search(TARGET_NAME)
    target_ids = [t["target_chembl_id"] for t in targets if TARGET_NAME.lower() in str(t).lower()]

    records = []
    for target_id in target_ids:
        acts = activity.filter(target_chembl_id=target_id, standard_type=STANDARD_TYPE)
        records.extend(list(acts))

    raw_df = pd.DataFrame(records)
    raw_df.to_csv(RAW_FILE, index=False)

print(raw_df.shape)
raw_df.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
(2423, 47)


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,active,6720436,[],CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,F,None,None,BAO_0000188,...,Mycobacterium tuberculosis,Protein RecA,1773,None,None,EC50,uM,UO_0000065,None,18.6
1,None,active,6720437,[],CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,F,None,None,BAO_0000188,...,Mycobacterium tuberculosis,Protein RecA,1773,None,None,EC50,uM,UO_0000065,None,9.87
2,None,active,6720438,[],CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,F,None,None,BAO_0000188,...,Mycobacterium tuberculosis,Protein RecA,1773,None,None,EC50,uM,UO_0000065,None,0.274
3,None,active,6720439,[],CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,F,None,None,BAO_0000188,...,Mycobacterium tuberculosis,Protein RecA,1773,None,None,EC50,uM,UO_0000065,None,20.0
4,None,active,6720440,[],CHEMBL1794426,PUBCHEM_BIOASSAY: Fluorescence Cell-Free Homog...,F,None,None,BAO_0000188,...,Mycobacterium tuberculosis,Protein RecA,1773,None,None,EC50,uM,UO_0000065,None,4.88


## 3. Curate EC50 data and assign binary labels

In [5]:
required = ["canonical_smiles", "standard_value", "standard_units", "molecule_chembl_id"]
missing = [c for c in required if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw_df.copy()
df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["canonical_smiles", "standard_value"])

# Keep nM records when unit information exists
if "standard_units" in df.columns:
    df = df[df["standard_units"].astype(str).str.lower().isin(["nm", "nanomolar"]) | df["standard_units"].isna()]

df = df[df["standard_value"] > 0].copy()
df["pEC50"] = 9 - np.log10(df["standard_value"])
df["bioactivity_class"] = np.where(df["standard_value"] <= ACTIVE_THRESHOLD_NM, 1, 0)

# Remove duplicate molecules using the median EC50/pEC50 value
curated = (
    df.groupby(["molecule_chembl_id", "canonical_smiles"], as_index=False)
      .agg(
          EC50_nM=("standard_value", "median"),
          pEC50=("pEC50", "median"),
          bioactivity_class=("bioactivity_class", "max"),
      )
)

curated = curated.rename(columns={"molecule_chembl_id": "Name", "canonical_smiles": "SMILES"})
curated["Name"] = curated["Name"].astype(str)

print(curated.shape)
print(curated["bioactivity_class"].value_counts())
curated.head()

(2344, 5)
bioactivity_class
1    1493
0     851
Name: count, dtype: int64


,Name,SMILES,EC50_nM,pEC50,bioactivity_class
0,CHEMBL1076559,Oc1ccc(Nc2nc(-c3ccc4ccccc4c3)cs2)cc1,7300.0,5.136677,1
1,CHEMBL1077990,COC(=O)c1sc(-c2cccs2)cc1NC(=O)/C=C/C(=O)O,15200.0,4.818156,0
2,CHEMBL1093246,Oc1ccc(/N=C/c2ccc(O)c(O)c2)cc1,380000.0,3.420216,0
3,CHEMBL1098875,O=C1c2cc(F)ccc2-n2c1nc1ccccc1c2=O,380000.0,3.420216,0
4,CHEMBL1142,Nc1ccc(O)cc1,380000.0,3.420216,0


## 4. Export final curated dataset

In [6]:
CURATED_FILE = CURATED_DIR / "recA_curated_ec50_binary.csv"
curated.to_csv(CURATED_FILE, index=False)
print(f"Saved: {CURATED_FILE}")

Saved: /content/outputs/recA_chembl/curated/recA_curated_ec50_binary.csv


## Final output
`outputs/recA_chembl/curated/recA_curated_ec50_binary.csv`